In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


In [ ]:
# ======================
# 1. ПАРАМЕТРЫ
# ======================
DATA = Path("../data/daily_sales.csv")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 9


In [ ]:
data = pd.read_csv(DATA, parse_dates=["date"])


In [ ]:
print("Number of rows:", data.shape[0])
print("Columns:", list(data.columns))
print("date range:", data["date"].min(), "→", data["date"].max())


In [ ]:
data.info()
data.head()


In [ ]:
# Promo доля
print('promo rate:', round(data['promo'].mean(), 3))
print('stores:', sorted(data['store_id'].unique()))


In [ ]:
# Sales by store
plt.figure(figsize=(8, 4))
sns.boxplot(data=data, x='store_id', y='sales', palette='viridis', hue='store_id', legend=False)
plt.title('Продажи по магазинам', fontsize=13)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(FIG / 'sales_by_store.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Агрегат продаж по дням (все магазины)
daily = data.groupby("date", as_index=False).agg(
    sales=("sales", "sum"),
    promo=("promo", "mean"),
)
daily.head()


In [ ]:
# ======================
# 2. ВИЗУАЛИЗАЦИЯ РЯДА
# ======================
plt.figure(figsize=(12, 4))
plt.plot(daily["date"], daily["sales"], lw=0.9, color="#31688e", label="суммарные продажи")
plt.title("Дневные продажи (все магазины)", fontsize=14, pad=15)
plt.xlabel("Дата", fontsize=12)
plt.ylabel("Sales", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.savefig(FIG / "daily_sales_timeseries.png", dpi=120, bbox_inches="tight")
plt.savefig(FIG / "eda_sales_line.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ======================
# 3. ПРИЗНАКИ ИЗ ДАТЫ
# ======================
df = data.copy()
df["dayofweek"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df = df.sort_values("date").reset_index(drop=True)

feature_cols = ["store_id", "promo", "dayofweek", "month", "day"]
target = "sales"
df[feature_cols + [target]].head()


In [ ]:
# ======================
# 4. RANDOM SPLIT (плохо для времени)
# ======================
rng = np.random.default_rng(RANDOM_SEED)
idx = rng.permutation(len(df))
cut = int(len(df) * 0.75)
tr_idx, te_idx = idx[:cut], idx[cut:]

X = df[feature_cols]
y = df[target]

pipe = Pipeline([("sc", StandardScaler()), ("m", LinearRegression())])
pipe.fit(X.iloc[tr_idx], y.iloc[tr_idx])
pred_rand = pipe.predict(X.iloc[te_idx])

print("RANDOM split:")
print("  MAE ", round(mean_absolute_error(y.iloc[te_idx], pred_rand), 2))
print("  RMSE", round(mean_squared_error(y.iloc[te_idx], pred_rand) ** 0.5, 2))
print("  R2  ", round(r2_score(y.iloc[te_idx], pred_rand), 3))


In [ ]:
# ======================
# 5. TIME-BASED SPLIT (правильно)
# ======================
# Берём последние 25% дат как тест — без перемешивания будущего
dates_sorted = np.sort(df["date"].unique())
cut_date = dates_sorted[int(len(dates_sorted) * 0.75)]
print("cut_date:", cut_date)

tr_mask = df["date"] <= cut_date
te_mask = ~tr_mask

pipe_time = Pipeline([("sc", StandardScaler()), ("m", LinearRegression())])
pipe_time.fit(X.loc[tr_mask], y.loc[tr_mask])
pred_time = pipe_time.predict(X.loc[te_mask])

print("TIME split:")
print("  MAE ", round(mean_absolute_error(y.loc[te_mask], pred_time), 2))
print("  RMSE", round(mean_squared_error(y.loc[te_mask], pred_time) ** 0.5, 2))
print("  R2  ", round(r2_score(y.loc[te_mask], pred_time), 3))


In [ ]:
# ======================
# 6. RF НА TIME SPLIT
# ======================
rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=RANDOM_SEED)
rf.fit(X.loc[tr_mask], y.loc[tr_mask])
pred_rf = rf.predict(X.loc[te_mask])

print("RF TIME split:")
print("  MAE ", round(mean_absolute_error(y.loc[te_mask], pred_rf), 2))
print("  RMSE", round(mean_squared_error(y.loc[te_mask], pred_rf) ** 0.5, 2))
print("  R2  ", round(r2_score(y.loc[te_mask], pred_rf), 3))


In [ ]:
# Сравнение метрик
cmp = pd.DataFrame([
    {"split": "random", "model": "lin", "MAE": mean_absolute_error(y.iloc[te_idx], pred_rand)},
    {"split": "time", "model": "lin", "MAE": mean_absolute_error(y.loc[te_mask], pred_time)},
    {"split": "time", "model": "rf", "MAE": mean_absolute_error(y.loc[te_mask], pred_rf)},
])

plt.figure(figsize=(8, 5))
sns.barplot(data=cmp, x="split", y="MAE", hue="model", palette="viridis")
plt.title("MAE: random split vs time split", fontsize=14, pad=15)
plt.ylabel("MAE", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIG / "sales_split_compare.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Факт vs прогноз на time-test (агрегат по дням)
te = df.loc[te_mask].copy()
te["pred"] = pred_rf
daily_te = te.groupby("date").agg(sales=("sales", "sum"), pred=("pred", "sum"))

plt.figure(figsize=(12, 4))
plt.plot(daily_te.index, daily_te["sales"], label="факт", lw=1.2)
plt.plot(daily_te.index, daily_te["pred"], label="прогноз RF", lw=1.2, alpha=0.8)
plt.title("Time-test: факт vs прогноз (сумма по дням)", fontsize=14, pad=15)
plt.xlabel("Дата", fontsize=12)
plt.ylabel("Sales", fontsize=12)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(FIG / "sales_time_forecast.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
print("Random split завышает качество на временных данных (утечка будущего).")
print("Scaler/энкодеры — только на train-части таймлайна.")
